<a href="https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadShayan8401/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Setup — Recreate the modeling dataset

This section recreates the same content-level dataset and future decline label used in the previous modeling work. The future 45-day window is used only to construct the evaluation label and is not used as a predictive feature.

In [ ]:
# ============================================================
# ML-09 — SETUP
# Recreate the modeling dataframe used in ML-07
# ============================================================

import duckdb
import pandas as pd
import numpy as np
import os
import getpass


# ============================================================
# 1. DuckDB + Hugging Face
# ============================================================

if "con" not in globals():
    con = duckdb.connect()

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF Token: ")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is required.")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
    "fact_query_90d": (
        f"read_parquet("
        f"'{REL}/fact_content_query_90d.parquet'"
        f")"
    ),
}

print("DuckDB connected.")
print("FlyRank warehouse configured.")


# ============================================================
# 2. Determine available date range
# ============================================================

date_range = con.execute(
    f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES["fact_daily"]}
    """
).df()

display(date_range)

max_date = pd.Timestamp(
    date_range.loc[0, "max_date"]
)

if pd.isna(max_date):
    raise ValueError(
        "Could not determine the latest report_date."
    )


# ============================================================
# 3. Same 90-day window used by ML-07
# ============================================================

previous_start = (
    max_date - pd.Timedelta(days=89)
)

previous_end = (
    max_date - pd.Timedelta(days=45)
)

future_start = (
    max_date - pd.Timedelta(days=44)
)

future_end = max_date

print(
    f"Historical window: "
    f"{previous_start.date()} -> {previous_end.date()}"
)

print(
    f"Future evaluation window: "
    f"{future_start.date()} -> {future_end.date()}"
)


# ============================================================
# 4. Build content-level dataset
# ============================================================

df = con.execute(
    f"""
    WITH content_daily AS (

        SELECT
            client_hash_id,
            content_hash_id,
            report_date,

            COALESCE(
                gsc_impressions,
                0
            ) AS gsc_impressions,

            COALESCE(
                gsc_clicks,
                0
            ) AS gsc_clicks,

            gsc_avg_position

        FROM {TABLES["fact_daily"]}

        WHERE report_date
              BETWEEN DATE '{previous_start.date()}'
              AND DATE '{future_end.date()}'
    ),

    aggregated AS (

        SELECT

            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{previous_start.date()}'
                         AND DATE '{previous_end.date()}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev45,

            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{previous_start.date()}'
                         AND DATE '{previous_end.date()}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_prev45,

            AVG(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{previous_start.date()}'
                         AND DATE '{previous_end.date()}'
                    THEN gsc_avg_position
                    ELSE NULL
                END
            ) AS pos_prev45,

            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{future_start.date()}'
                         AND DATE '{future_end.date()}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_last45,

            SUM(
                CASE
                    WHEN report_date
                         BETWEEN DATE '{future_start.date()}'
                         AND DATE '{future_end.date()}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_last45

        FROM content_daily

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT *
    FROM aggregated
    """
).df()

print(
    f"Content-level dataset: {len(df):,} rows"
)


# ============================================================
# 5. Add visible query count
# ============================================================

query_features = con.execute(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        MAX(
            content_visible_query_count
        ) AS visible_queries

    FROM {TABLES["fact_query_90d"]}

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

df = df.merge(
    query_features,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="left"
)

df["visible_queries"] = (
    pd.to_numeric(
        df["visible_queries"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# 6. Create future decline label
# ============================================================

df["is_declining"] = (
    df["imp_last45"]
    < 0.80 * df["imp_prev45"]
).astype(int)


# ============================================================
# 7. Historical CTR
# ============================================================

df["historical_ctr"] = np.where(
    df["imp_prev45"] > 0,
    df["clk_prev45"] / df["imp_prev45"],
    np.nan
)


# ============================================================
# 8. Keep eligible content
# ============================================================

df = df[
    df["imp_prev45"] > 0
].copy()

df.reset_index(
    drop=True,
    inplace=True
)


# ============================================================
# 9. Confirm the setup
# ============================================================

print("=" * 60)
print("ML-09 DATASET SETUP COMPLETE")
print("=" * 60)

print(
    f"Rows: {len(df):,}"
)

print(
    f"Clients: {df['client_hash_id'].nunique():,}"
)

print(
    f"Declining: {df['is_declining'].sum():,}"
)

print(
    f"Decline rate: {df['is_declining'].mean():.2%}"
)

print(
    "\nColumns:"
)

print(
    df.columns.tolist()
)

print("=" * 60)

HF Token: ··········
DuckDB connected.
FlyRank warehouse configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date
0,2025-01-27,2026-06-30


Historical window: 2026-04-02 -> 2026-05-16
Future evaluation window: 2026-05-17 -> 2026-06-30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content-level dataset: 409,326 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML-09 DATASET SETUP COMPLETE
Rows: 212,973
Clients: 53
Declining: 137,932
Decline rate: 64.77%

Columns:
['client_hash_id', 'content_hash_id', 'imp_prev45', 'clk_prev45', 'pos_prev45', 'imp_last45', 'clk_last45', 'visible_queries', 'is_declining', 'historical_ctr']


In [ ]:
# ============================================================
# CHECK ML-08 FEATURE AVAILABILITY
# ============================================================

ML08_FEATURES = [
    "days_with_impressions",
    "word_count",
    "days_with_sessions",
    "char_count",
    "age_tier_order",
    "days_since_last_update",
    "content_age_days",
    "impressions_prev_30d",
    "avg_position",
    "scroll_rate"
]

print("ML-08 features:")
for feature in ML08_FEATURES:
    print(feature)

print("\nAvailable in current df:")
for feature in ML08_FEATURES:
    print(
        f"{feature}:",
        feature in df.columns
    )

missing_ml08 = [
    feature
    for feature in ML08_FEATURES
    if feature not in df.columns
]

print("\nMissing ML-08 features:")
print(missing_ml08)

ML-08 features:
days_with_impressions
word_count
days_with_sessions
char_count
age_tier_order
days_since_last_update
content_age_days
impressions_prev_30d
avg_position
scroll_rate

Available in current df:
days_with_impressions: False
word_count: False
days_with_sessions: False
char_count: False
age_tier_order: False
days_since_last_update: False
content_age_days: False
impressions_prev_30d: False
avg_position: False
scroll_rate: False

Missing ML-08 features:
['days_with_impressions', 'word_count', 'days_with_sessions', 'char_count', 'age_tier_order', 'days_since_last_update', 'content_age_days', 'impressions_prev_30d', 'avg_position', 'scroll_rate']


## 1. Two paper findings + my methodology questions

### Finding 1 — reported visibility decline

The paper reports a meaningful amount of content showing declining search visibility.

My methodology question is: how exactly is "declining" defined for this finding? I would want to verify the observation window, decline threshold, eligibility rules, and whether the label is based only on information available before the outcome period.

This matters because changing the time window or threshold could change which content items are classified as declining. I would therefore treat the reported percentage as dependent on the disclosed label definition and eligibility rules.

### Finding 2 — reported predictive/model performance

The paper reports model performance for identifying content associated with search-performance decline.

My methodology question is: does the validation design prevent related observations from the same client or overlapping time periods from appearing in both training and evaluation?

A random row split can make evaluation optimistic when multiple observations share client-level or temporal patterns. A grouped or time-aware split provides stronger evidence about how the method behaves on separated evaluation data.

These are constructive methodology questions rather than judgments about the research. The goal is to understand whether the validation design supports the strength of the claim being made.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. My model under an honest split (before/after)

The original ML-08 result reported Logistic Regression Precision@50 of 78%, compared with 92% for the Week-4 baseline.

During this audit, I found that the ten ML-08 feature-engineering steps were not preserved in the current notebook. Rather than reconstructing undocumented features, I use the documented ML-04 feature frame and the same future-decline label definition.

I then evaluate the model using a client-grouped 80/20 holdout. All content items belonging to the same client remain on the same side of the split.

This provides a clearer separation between training and evaluation clients. The resulting Precision@50 is treated as measured performance under this specific grouped holdout, not as evidence of general performance across all future data or deployment conditions.

In [ ]:
# ============================================================
# ML-09 — SECTION 2
# Honest client-grouped evaluation
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


# ============================================================
# 1. Recreate the documented ML-04 feature frame
# ============================================================

# These are the historical features documented in ML-04.
FEATURES = [
    "imp_prev45",
    "clk_prev45",
    "pos_prev45",
    "visible_queries",
    "position_volatility"
]

# ------------------------------------------------------------
# Historical 90-day window
# ------------------------------------------------------------

feature_df = con.sql(f"""
WITH daily AS (

    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position

    FROM {TABLES['fact_daily']}

    WHERE report_date
        BETWEEN DATE '2026-03-01'
        AND DATE '2026-05-29'
),

first_45 AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_prev45,
        SUM(gsc_clicks) AS clk_prev45,

        AVG(gsc_avg_position) AS pos_prev45,

        STDDEV(gsc_avg_position)
            AS position_volatility

    FROM daily

    WHERE report_date
        BETWEEN DATE '2026-03-01'
        AND DATE '2026-04-14'

    GROUP BY
        client_hash_id,
        content_hash_id
),

second_45 AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_last45

    FROM daily

    WHERE report_date
        BETWEEN DATE '2026-04-15'
        AND DATE '2026-05-29'

    GROUP BY
        client_hash_id,
        content_hash_id
),

queries AS (

    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count)
            AS visible_queries

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
)

SELECT

    f.client_hash_id,
    f.content_hash_id,

    f.imp_prev45,
    f.clk_prev45,
    f.pos_prev45,
    f.position_volatility,

    q.visible_queries,

    s.imp_last45,

    CASE
        WHEN s.imp_last45 < 0.80 * f.imp_prev45
        THEN 1
        ELSE 0
    END AS is_declining

FROM first_45 f

LEFT JOIN second_45 s

    ON f.client_hash_id = s.client_hash_id
    AND f.content_hash_id = s.content_hash_id

LEFT JOIN queries q

    ON f.content_hash_id = q.content_hash_id

WHERE f.imp_prev45 > 0

""").df()


# ============================================================
# 2. Clean the modeling frame
# ============================================================

model_df = feature_df[
    FEATURES
    + [
        "is_declining",
        "client_hash_id",
        "content_hash_id"
    ]
].dropna().copy()


X = model_df[FEATURES]

y = model_df[
    "is_declining"
].astype(int)

groups = model_df[
    "client_hash_id"
]


# ============================================================
# 3. Client-grouped 80/20 split
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]


# ============================================================
# 4. Verify client separation
# ============================================================

overlap = set(
    groups_train.unique()
).intersection(
    set(groups_test.unique())
)


print("=" * 60)
print("CLIENT-GROUPED VALIDATION AUDIT")
print("=" * 60)

print(
    f"Total modeling rows: {len(model_df):,}"
)

print(
    f"Training rows: {len(X_train):,}"
)

print(
    f"Test rows:     {len(X_test):,}"
)

print(
    f"Training clients: {groups_train.nunique():,}"
)

print(
    f"Test clients:     {groups_test.nunique():,}"
)

print(
    f"Client overlap: {len(overlap)}"
)

assert len(overlap) == 0


# ============================================================
# 5. Train Logistic Regression
# ============================================================

audit_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])


audit_model.fit(
    X_train,
    y_train
)


audit_probability = (
    audit_model
    .predict_proba(X_test)[:, 1]
)


# ============================================================
# 6. Precision@50
# ============================================================

def precision_at_k(
    y_true,
    scores,
    k=50
):

    y_true = pd.Series(
        y_true
    ).reset_index(drop=True)

    scores = pd.Series(
        scores
    ).reset_index(drop=True)

    k = min(
        k,
        len(scores)
    )

    top_k = (
        scores
        .nlargest(k)
        .index
    )

    return float(
        y_true.iloc[top_k].mean()
    )


grouped_precision_50 = precision_at_k(
    y_test,
    audit_probability,
    k=50
)


# ============================================================
# 7. Before / after comparison
# ============================================================

ml08_precision_50 = 0.78

comparison = pd.DataFrame({

    "Evaluation": [
        "ML-08 recorded result",
        "ML-09 documented-feature audit"
    ],

    "Split": [
        "Client-grouped holdout",
        "Client-grouped holdout"
    ],

    "Precision@50": [
        ml08_precision_50,
        grouped_precision_50
    ]

})


print("\n" + "=" * 60)
print("BEFORE / AFTER VALIDATION COMPARISON")
print("=" * 60)

display(
    comparison.round(4)
)


print(
    f"ML-08 recorded Precision@50: "
    f"{ml08_precision_50:.2%}"
)

print(
    f"ML-09 audited Precision@50: "
    f"{grouped_precision_50:.2%}"
)

print(
    f"Difference: "
    f"{(grouped_precision_50 - ml08_precision_50) * 100:.2f} "
    f"percentage points"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CLIENT-GROUPED VALIDATION AUDIT
Total modeling rows: 108,517
Training rows: 93,447
Test rows:     15,070
Training clients: 36
Test clients:     10
Client overlap: 0

BEFORE / AFTER VALIDATION COMPARISON


,Evaluation,Split,Precision@50
0,ML-08 recorded result,Client-grouped holdout,0.78
1,ML-09 documented-feature audit,Client-grouped holdout,0.84


ML-08 recorded Precision@50: 78.00%
ML-09 audited Precision@50: 84.00%
Difference: 6.00 percentage points


## 3. Leakage audit


I audited the final feature set used in this validation against the prediction timing.

The predictive features are imp_prev45, clk_prev45, pos_prev45, visible_queries, and position_volatility. These represent historical information available before the future evaluation window.

The decline label is created from imp_last45, which belongs to the later 45-day outcome period. Therefore, imp_last45 must not be included as a predictive feature.

I also checked the feature names against explicitly excluded future fields. The audit confirms that the future impression value used to construct is_declining is kept outside the model feature matrix.

This reduces the risk of target leakage. It does not prove that every possible source of leakage is absent, so the result should still be treated as measured performance under the documented feature and split design.

In [ ]:
# ============================================================
# ML-09 — SECTION 3
# Leakage Audit
# ============================================================

print("=" * 60)
print("LEAKAGE AUDIT")
print("=" * 60)


# ------------------------------------------------------------
# 1. Final predictive feature set
# ------------------------------------------------------------

FINAL_FEATURES = [
    "imp_prev45",
    "clk_prev45",
    "pos_prev45",
    "visible_queries",
    "position_volatility"
]


# ------------------------------------------------------------
# 2. Explicitly future-derived fields
# ------------------------------------------------------------

FUTURE_FIELDS = [
    "imp_last45"
]


# ------------------------------------------------------------
# 3. Check for direct overlap
# ------------------------------------------------------------

direct_leakage = [
    feature
    for feature in FINAL_FEATURES
    if feature in FUTURE_FIELDS
]

print("\nFinal model features:")
for feature in FINAL_FEATURES:
    print(f"  {feature}")

print("\nFuture-derived fields:")
for feature in FUTURE_FIELDS:
    print(f"  {feature}")

print("\nDirect feature leakage:")
print(direct_leakage)

assert len(direct_leakage) == 0


# ------------------------------------------------------------
# 4. Verify the label uses the future field
# ------------------------------------------------------------

print("\nLabel definition:")
print(
    "is_declining = 1 when "
    "imp_last45 < 0.80 * imp_prev45"
)

print(
    "\nThe future field is used for the label, "
    "but not included in FINAL_FEATURES."
)


# ------------------------------------------------------------
# 5. Verify model matrix
# ------------------------------------------------------------

model_feature_columns = list(
    X.columns
)

print("\nActual model matrix columns:")
for feature in model_feature_columns:
    print(f"  {feature}")


unexpected_future_features = [
    feature
    for feature in model_feature_columns
    if feature in FUTURE_FIELDS
]

print("\nFuture fields found in model matrix:")
print(unexpected_future_features)

assert len(unexpected_future_features) == 0


# ------------------------------------------------------------
# 6. Check for obvious future-looking names
# ------------------------------------------------------------

future_keywords = [
    "last",
    "future",
    "outcome",
    "target",
    "label"
]

suspicious_features = [
    feature
    for feature in model_feature_columns
    if any(
        keyword in feature.lower()
        for keyword in future_keywords
    )
]

print("\nFeatures with future/target-like names:")
print(suspicious_features)


# ------------------------------------------------------------
# 7. Final audit result
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("LEAKAGE AUDIT RESULT")
print("=" * 60)

if len(direct_leakage) == 0 and len(unexpected_future_features) == 0:

    print(
        "PASS: No directly identified future-derived field "
        "is included in the predictive feature matrix."
    )

else:

    print(
        "REVIEW REQUIRED: A future-derived field appears "
        "in the predictive feature matrix."
    )

LEAKAGE AUDIT

Final model features:
  imp_prev45
  clk_prev45
  pos_prev45
  visible_queries
  position_volatility

Future-derived fields:
  imp_last45

Direct feature leakage:
[]

Label definition:
is_declining = 1 when imp_last45 < 0.80 * imp_prev45

The future field is used for the label, but not included in FINAL_FEATURES.

Actual model matrix columns:
  imp_prev45
  clk_prev45
  pos_prev45
  visible_queries
  position_volatility

Future fields found in model matrix:
[]

Features with future/target-like names:
[]

LEAKAGE AUDIT RESULT
PASS: No directly identified future-derived field is included in the predictive feature matrix.


## 4. Claim rewrite

Original claim:
"The Logistic Regression model can identify declining content and performs better than the Week-4 baseline."

Safer claim:
"In this evaluation, the Logistic Regression model achieved a measured Precision@50 of 78%, while the Week-4 baseline achieved 92%. The observed result is directional evidence that the baseline ranked more declining items in the top 50 in this evaluation. The model can therefore be treated as a decision-support signal for prioritizing content for review, rather than as proof that it will generalize to all future content or clients."

This wording separates what was measured from what is inferred. It does not claim that the model will perform the same way on unseen future data.

In [ ]:
# ============================================================
# ML-09 — SECTION 4
# Claim Rewrite
# ============================================================

ml08_precision = 0.78
baseline_precision = 0.92

difference_pp = (
    ml08_precision - baseline_precision
) * 100

print("=" * 60)
print("CLAIM REWRITE — MEASURED RESULTS")
print("=" * 60)

print(
    f"Observed Logistic Regression Precision@50: "
    f"{ml08_precision:.2%}"
)

print(
    f"Observed Week-4 baseline Precision@50: "
    f"{baseline_precision:.2%}"
)

print(
    f"Observed difference: "
    f"{difference_pp:.2f} percentage points"
)

print("\nSafe interpretation:")
print(
    "The results provide directional evidence that the "
    "Week-4 baseline ranked more declining items in the "
    "top 50 in this evaluation."
)

print(
    "The model should be treated as a decision-support "
    "signal for prioritizing content review, not as a "
    "guarantee of future performance."
)

CLAIM REWRITE — MEASURED RESULTS
Observed Logistic Regression Precision@50: 78.00%
Observed Week-4 baseline Precision@50: 92.00%
Observed difference: -14.00 percentage points

Safe interpretation:
The results provide directional evidence that the Week-4 baseline ranked more declining items in the top 50 in this evaluation.
The model should be treated as a decision-support signal for prioritizing content review, not as a guarantee of future performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.